In [ ]:
import pandas as pd
import numpy as np
import time
import utils

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    roc_auc_score,
    precision_recall_curve,
    auc, average_precision_score
)
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Baseline Imports
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from catboost import CatBoostClassifier, CatBoostRegressor

import torch

from tabpfn import TabPFNClassifier, TabPFNRegressor
#from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier, AutoTabPFNRegressor




In [ ]:
files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

for file in files:
    print(file)
    print(file.split("/")[-1].strip(".txt") + "_results.csv")

In [ ]:
test =  pd.read_csv("data/INI_DataSet.txt", sep='\t')

drugs = ["CAB", "RAL", "EVG", "DTG", "BIC"]

for drug in drugs:
    print(drug + ": " , test[drug].nunique())

In [ ]:
input_file = files[0]
#thresholds defined by the database for the classes of "susceptible", "partly susceptible", and "resistant"
thresholds = [
    [3, 15],  # FPV
    [3, 15],  # ATV
    [3, 15],  # IDV
    [9, 55],  # LPV
    [3, 6],  # NFV
    [3, 15],  # SQV
    [2, 8],  # TPV
    [10, 90],  # DRV
    [5, 25],  # X3TC
    [2, 6],  # ABC
    [3, 15],  # AZT
    [1.5, 3],  # D4T
    [1.5, 3],  # DDI
    [1.5, 3],  # TDF
    [3, 10],  # EFV
    [3, 10],  # NVP
    [3, 10],  # ETR
    [3, 10],  # RPV
    [2.5, 10],  # BIC
    [4, 13],  # DTG
    [2.5, 10],  # EVG - upper threshold guessed
    [1.5, 10]  # RAL - upper threshold guessed
]

# Define row and column names
index = ["FPV", "ATV", "IDV", "LPV", "NFV", "SQV", "TPV", "DRV",
         "3TC", "ABC", "AZT", "D4T", "DDI", "TDF",
         "EFV", "NVP", "ETR", "RPV", "BIC", "DTG", "EVG", "RAL"]
columns = ["lower", "upper"]

# Create DataFrame
cutoff_df = pd.DataFrame(thresholds, index=index, columns=columns)

# Reading in and processing high quality File

df = pd.read_csv(input_file, sep='\t')
#print(df)
df = df.iloc[:,1:-1]
print(df)

#Checking how much data is available for each drug
#print(df.loc[:,"FPV":"DRV"].count())

#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

#creating the one hot encoding for the features
enc = OneHotEncoder(handle_unknown='error')

enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])


#going through the drugs and splitting them to test and training depending on the drug

results = pd.DataFrame(columns=["Drug",
                                "RMSE",
                                "AUC ROC MC",
                                "AUC PRC MC",
                                "AUC ROC BI",
                                "AUC PRC BI",
                                "Time"])

print(input_file)

for drug in drugs:
    #print(drug)
    tmp_drugs = drugs.copy()
    #print(tmp_drugs)
    tmp_drugs.remove(drug)
    #print(tmp_drugs)
    last_col = list(df.columns)[-1]
    dataframe = df.drop(tmp_drugs, axis=1)

    #print(dataframe.head())

    dataframe = dataframe.dropna()


    '''    # encoding the levels of susceptibility as 0 for susceptible, 1 as resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 1
    '''

    # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completely resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "upper"], drug + "_level"] = 2
    dataframe.loc[(dataframe[drug] >= cutoff_df.loc[drug, "lower"]) & (
                dataframe[drug] < cutoff_df.loc[drug, "upper"]), drug + "_level"] = 1

    #print(dataframe.head())

    X, y = dataframe.drop([drug, drug + "_level"], axis=1), np.array(dataframe[drug + "_level"])

    print(y)

    print(X.shape)
    if X.shape[0] == 0:
        continue

    X_trafo = enc.transform(X).toarray()

    #print(X)


In [ ]:
# Reading in and processing high quality File
df = pd.read_csv(input_file, sep='\t')
#print(df)
df = df.iloc[:,1:-1]
#print(df2)

#Checking how much data is available for each drug
#print(df.loc[:,"FPV":"DRV"].count())

#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

#creating the one hot encoding for the features
enc = OneHotEncoder(handle_unknown='error')

enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])


#going through the drugs and splitting them to test and training depending on the drug

results = pd.DataFrame(columns=["Drug",
                            "Samples",
                            "AUC ROC",
                            "Time",
                            "AUC RF",
                            "AUC XGB",
                            "AUC CatB"])



for drug in drugs:
    #print(drug)
    tmp_drugs = drugs.copy()
    #print(tmp_drugs)
    tmp_drugs.remove(drug)
    #print(tmp_drugs)
    last_col = list(df.columns)[-1]
    dataframe = df.drop(tmp_drugs, axis=1)

    #print(dataframe.head())

    dataframe = dataframe.dropna()

    if drug not in utils.THRESHOLD_INDICES:
        results = pd.concat([pd.DataFrame([[drug, dataframe.shape[0], None, None, None,
                                            None, None]], columns=results.columns),
                             results], ignore_index=True)
        continue

    # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completly resistant

    y = utils.get_classes(dataframe, drug, mode="multiclass")

    #print(dataframe.head())

    X = dataframe.drop([drug], axis=1)

    print(X)

    X_trafo = enc.transform(X).toarray()

    print(X_trafo.shape)

    print(y)
    X_train, X_test, y_train, y_test = train_test_split(X_trafo, y, test_size=0.33, random_state=42)


In [ ]:
"""This script calculates the ROC AUC for the prediction of TabPFN, Random Forest, XGBoost, and
CatBoost and saves time in a file for all drugs in the stanford database file"""

# Setup Imports
import pandas as pd
import numpy as np
import time

import sys
import os

#sys.path.append(os.path.abspath('..'))

import utils

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr

from sklearn.preprocessing import OneHotEncoder

# Baseline Imports

from tabpfn import TabPFNClassifier


# table for the encoding of the resistance testing into three classes: "susceptible", "intermediate-level resistant", "high-level resistant" with lower and upper thresholds

def running_models(input_file, output_file):


    # Reading in and processing high quality File
    df = pd.read_csv(input_file, sep='\t')

    #removing index and summary column
    df = df.iloc[:,1:-1]


    #list of current drugs of the dataset
    drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

    #creating the one hot encoding for the features
    enc = OneHotEncoder(handle_unknown='error')

    enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])

    #result dataframe
    results = pd.DataFrame(columns=["Drug",
                                    "Samples",
                                    "Accuracy",
                                    "Pearson",
                                    "F1"
                                    "AUC PRC",
                                    "AUC ROC",
                                    "Time"])



    for drug in drugs:

        tmp_drugs = drugs.copy().remove(drug)

        #getting labels of only needed drug
        #dataframe = df.drop(tmp_drugs, axis=1)

        #print(dataframe)

        dataframe = df.dropna(subset=[drug])

        #If no thresholds for drug available no prediction possible
        if drug not in utils.THRESHOLD_INDICES:
            results = pd.concat([pd.DataFrame([[drug, dataframe.shape[0], None, None, None,
                                                None, None]], columns=results.columns),
                                 results], ignore_index=True)
            continue


        # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completly resistant
        y = utils.get_classes(dataframe, drug, mode="multiclass")


        X = dataframe.drop([drug], axis=1)



        #X_trafo = enc.transform(X).toarray()



    #results.to_csv(output_file)

def main():


    files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

    for file in files:
        running_models(file, "output/" + (file.split("/")[-1].strip(".txt") + "multilabel_results.csv"))

if __name__ == '__main__':
    main()

In [ ]:
files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

drug = "testdrug"

for file in files:
        print("../prediction_results/" + (file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "Multilabel_prediction") + ".csv")


In [ ]:
drugs = ['FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV']
drug = "FPV"

tmp_drugs = drugs.copy()
tmp_drugs.remove(drug)

print(tmp_drugs)
print(drugs)

In [ ]:
import numpy as np

import utils

drugs = ['FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV']
drug = "FPV"

files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

input_file = files[0]

print(input_file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "AutoTabPFN_results")

#utils.save_results( np.array([0]), np.array([0]), label= (input_file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "AutoTabPFN_results"))

In [ ]:
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

#x = np.array([0.9, 1.1, 1.15, 1.0, 1.1, 1.8, 1.9, 1.95, 2.8, 3.4, 3.5, 4.7, 6.1])
#y= np.array([0.125, 0.145, 0.18, 0.23, 0.24, 0.1, 0.11, 0.2, 0.16, 0.12, 0.13, 0.095, 0])

x = np.array([0.9, 1.1, 1.15, 1.0, 1.1, 1.8, 1.9, 1.95, 2.8, 3.4, 3.5, 4.7])
y= np.array([0.125, 0.145, 0.18, 0.23, 0.24, 0.1, 0.11, 0.2, 0.16, 0.12, 0.13, 0.095])

X = x.reshape(-1, 1)

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Get slope and intercept
slope = model.coef_[0]
intercept = model.intercept_

print(f"y = {slope:.4f}x + {intercept:.4f}")

# Predict for plotting
x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
y_pred = model.predict(x_range)

# Plot
plt.scatter(x, y, color='red', label='Data points')
plt.plot(x_range, y_pred, color='blue', label='Fit: y={:.3f}x+{:.3f}'.format(slope, intercept))
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

print(r2_score(y, model.predict(X)))


In [ ]:
import pandas as pd
from scipy.io import arff

# code
arff_file = arff.loadarff('./data/Other_datasets/yeast.arff')


df = pd.DataFrame(arff_file[0])


df.head()

In [ ]:
import utils
import numpy as np
import pandas as pd

input_file = r"./data/PI_DataSet.txt"

# Reading in and processing high quality File
df = pd.read_csv(input_file, sep='\t')

#removing index and summary column
df = df.iloc[:,1:-1]


#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

drug = drugs[0]


#if type(drug) != list:
#   drug = [drug]

print(type(drugs))
print(drug)

print(df)

y = utils.get_classes(df, drugs, mode="multiclass")



print(y)

X = df.drop(drugs, axis=1)


print(X)

In [18]:
from sklearn.datasets import make_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import shuffle
import numpy as np

from tabpfn import TabPFNClassifier

X, y1 = make_classification(n_samples=10, n_features=100,
                            n_informative=30, n_classes=3,
                            random_state=1)
y2 = shuffle(y1, random_state=1)
y3 = shuffle(y1, random_state=2)
Y = np.vstack((y1, y2, y3)).T
n_samples, n_features = X.shape # 10,100
n_outputs = Y.shape[1] # 3
n_classes = 3

clf = TabPFNClassifier()

print()

#forest = RandomForestClassifier(random_state=1)
multi_target_forest = MultiOutputClassifier(clf, n_jobs=2)
y_pred = multi_target_forest.fit(X, Y).predict(X)

In [19]:
print(y_pred)

[[2 2 0]
 [1 2 1]
 [2 1 0]
 [0 0 2]
 [0 2 1]
 [0 0 2]
 [1 1 0]
 [1 1 1]
 [0 0 2]
 [2 0 0]]


In [28]:
#print(y1)

y_true= y1

#y_pred = Y


if y_true.shape[0] != y_pred.shape[0]:
    raise Exception("True labels do not match predicted labels")

#splt = label.split('/')[:-1]
#sub_filepath = '/'.join(splt)

#print(sub_filepath)

#Path(path + sub_filepath).mkdir(parents=True, exist_ok=True)

data = {"True": y_true}

for i, column in enumerate(y_pred.T):
    data.update( {str(i): column })

#df = pd.DataFrame(data)

df2 = pd.DataFrame(Y)

#f = df2.add_prefix("Pred_")

y_pred = pd.DataFrame(y_pred, columns=["0", "1", "2"])
y_true = pd.DataFrame(Y, columns=["0", "1", "2"])

y_pred_new = y_pred.add_prefix("Pred_")
y_true_new = y_true.add_prefix("True_")

#y_true_new["True_0"] = 4.0

df = pd.concat([y_true_new, y_pred_new], sort=False, axis = 1)

#print(f)
print(y_pred)
print(y_true)

print(df)
#print(df)

   0  1  2
0  2  2  0
1  1  2  1
2  2  1  0
3  0  0  2
4  0  2  1
5  0  0  2
6  1  1  0
7  1  1  1
8  0  0  2
9  2  0  0
   0  1  2
0  2  2  0
1  1  2  1
2  2  1  0
3  0  0  2
4  0  2  1
5  0  0  2
6  1  1  0
7  1  1  1
8  0  0  2
9  2  0  0
   True_0  True_1  True_2  Pred_0  Pred_1  Pred_2
0       2       2       0       2       2       0
1       1       2       1       1       2       1
2       2       1       0       2       1       0
3       0       0       2       0       0       2
4       0       2       1       0       2       1
5       0       0       2       0       0       2
6       1       1       0       1       1       0
7       1       1       1       1       1       1
8       0       0       2       0       0       2
9       2       0       0       2       0       0


In [14]:
import pandas as pd

files = [r"./data/PI_DataSet.txt", r"./data/INI_DataSet.txt", r"./data/NRTI_DataSet.txt", r"./data/NNRTI_DataSet.txt"]

for file in files:
    #running_models(file, "../output/" + (file.split("/")[-1].strip(".txt") + "_multilabel_results.csv"))

    # Reading in and processing high quality File
    df = pd.read_csv(file, sep='\t')

    # removing index and summary column
    df = df.iloc[:, 1:-1]

    # list of current drugs of the dataset
    drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]



    unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

    if len(unusable_drugs) > 0:
        df.drop(columns=unusable_drugs, inplace=True)

        drugs = [drug for drug in drugs if drug not in unusable_drugs]

    print(file.split("/")[-1].split("_")[0] + "_results/" + file.split("/")[-1].split("_")[0] + "_Binary_Relevance_MOC_prediction")

PI_results/PI_Binary_Relevance_MOC_prediction
INI_results/INI_Binary_Relevance_MOC_prediction
NRTI_results/NRTI_Binary_Relevance_MOC_prediction
NNRTI_results/NNRTI_Binary_Relevance_MOC_prediction


In [ ]:
"""Implementation of the Binary relevance Multilable prediction algorithm using TabPFN and the HIV drug resistance dataset
as an example"""

# Setup Imports
import pandas as pd
import numpy as np
import time

import sys
import os


import utils

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier

from scipy.stats import pearsonr

from sklearn.preprocessing import OneHotEncoder

# Baseline Imports

from tabpfn import TabPFNClassifier


class BinaryRelevanceTabPFN():

    def predict(x, y):
        """
        #result dataframe
        results = pd.DataFrame(columns=["Drug",
                                        "Samples",
                                        "Accuracy",
                                        "Pearson",
                                        "F1",
                                        "AUC PRC",
                                        "AUC ROC",
                                        "Time"])



        for drug in drugs:
            print(input_file.split("/")[1].split("_")[0] + ": " + drug)
            tmp_drugs = drugs.copy().remove(drug)

            #getting labels of only needed drug
            #dataframe = df.drop(tmp_drugs, axis=1)

            dataframe = df.dropna(subset=[drug])

            #If no thresholds for drug available no prediction possible
            if drug not in utils.THRESHOLD_INDICES:
                results = pd.concat([pd.DataFrame([[drug, dataframe.shape[0], None, None, None,
                                                    None, None, None]], columns=results.columns),
                                     results], ignore_index=True)
                continue


            # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completly resistant
            y = utils.get_classes(dataframe, drug, mode="multiclass")


            X = dataframe.drop([drug], axis=1)



            #X_trafo = enc.transform(X).toarray()


            #----------------------------------------------------------------------------------------------------------------
            #Training


            #getting train test split
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

            #Timing TabPFN
            start_time = time.time()

            # Train and evaluate TabPFN
            y_pred = TabPFNClassifier(random_state=42, ignore_pretraining_limits=True).fit(X_train, y_train).predict_proba(X_test)

            taken_time = time.time() - start_time

            y_pred_class = np.argmax(y_pred, axis=1)

            #--------------------------------------------------------------------------------------------------------------------
            # Evaluation metrics

            #Accuracy:
            scores = {"Accuracy": accuracy_score(y_test, y_pred_class)}

            #Person coefficient:
            scores.update({"Pearson": pearsonr(y_test, y_pred_class)[0]})

            #F1 score:
            scores.update({"F1": f1_score(y_test, y_pred_class, average="micro")})

            # Calculate PRC AUC
            scores.update({"AUC PRC" : utils.prc_auc_score(y_test, y_pred, multiclass="ovr")})
            #print(f"TabPFN PRC AUC: {score_prc:.4f}")

            # Calculate ROC AUC (handles both binary and multiclass)
            scores.update({ "AUC ROC": roc_auc_score(y_test, y_pred if len(np.unique(y)) > 2 else y_pred[:, 1], multi_class='ovr')})
            #print(f"TabPFN ROC AUC: {score_roc:.4f}")


            #saving the resulting statistics
            results = pd.concat([pd.DataFrame([[
                                                drug,
                                                X.shape[0],
                                                scores["Accuracy"],
                                                scores["Pearson"],
                                                scores["F1"],
                                                scores["AUC PRC"],
                                                scores["AUC ROC"],
                                                taken_time
                                            ]], columns=results.columns), results], ignore_index=True)


            """




        pass

        #saving results:
        #utils.save_results(y_pred, y_test, label= (input_file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "Multilabel_prediction"))


        #return results

def main():


    #files = [r"../data/PI_DataSet.txt", r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt", r"../data/NNRTI_DataSet.txt"]

    files=[r"./data/PI_DataSet.txt"]

    for file in files:
        #running_models(file, "../output/" + (file.split("/")[-1].strip(".txt") + "_multilabel_results.csv"))

        # Reading in and processing high quality File
        df = pd.read_csv(file, sep='\t')

        # removing index and summary column
        df = df.iloc[:, 1:-1]

        # list of current drugs of the dataset
        drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

        #Filtering out drugs with less than 10 labels present
        unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

        if len(unusable_drugs) > 0:
            df.drop(columns=unusable_drugs, inplace=True)

            drugs = [drug for drug in drugs if drug not in unusable_drugs]

        df.dropna(subset=drugs, inplace=True)


        # creating the one hot encoding for the features
        #enc = OneHotEncoder(handle_unknown='error')

        #enc.fit(df.loc[:, [drug for drug in list(df.columns) if drug.startswith("P")]])



        X = df.drop(drugs, axis=1)




        Y = utils.get_classes(df, drugs, mode="binary")

        X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)

        clf = TabPFNClassifier()


        multi_target_pfn = MultiOutputClassifier(clf, n_jobs=2)
        y_pred = multi_target_pfn.fit(X_train, y_train).predict(X_test)



        #BR = BinaryRelevanceTabPFN()


        #results = BR.predict(X, Y)

        y_pred_df = pd.DataFrame(y_pred, columns=drugs)

        y_test_df = pd.DataFrame(y_test, columns=drugs)


        utils.save_multilabel(y_pred_df, y_test_df, label= "PI_test", path="./")



if __name__ == '__main__':
    main()